<a href="https://colab.research.google.com/github/lucas-estanislau/pe-de-meia/blob/Teste-de-otimiza%C3%A7%C3%A3o-da-coleta/01_coleta_taxas_rendimento_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import requests
import zipfile
from io import BytesIO
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install xlrd

In [4]:
dados2007 = pd.read_excel("/content/drive/MyDrive/pe-de-meia/inep/TAXAS RENDIMENTO UF 2007.xls", engine="xlrd")
dados2007

,Unnamed: 0,Ministério da Educação,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 49,Unnamed: 50,Unnamed: 51,Unnamed: 52,Unnamed: 53,Unnamed: 54,Unnamed: 55,Unnamed: 56,Unnamed: 57,Unnamed: 58
0,NaN,Instituto Nacional de Est...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Taxas de...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Taxa de Aprovação - 2007,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"Taxa de Aprovação, segundo a Localização e a D...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
477,2007,Centro_Oeste,DF,Urbana,Total,96.4,89.1,86,89.2,92.1,...,3.4,0.8,3,1.8,9.6,7.4,6,0,0,8
478,Fonte: Mec/Inep/Deed.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
479,"Notas: 1) Inclui as taxas de aprovação, r...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
480,2) Inclui as taxas do Ensino F...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
caminho = "/content/drive/MyDrive/pe-de-meia/inep/TAXAS RENDIMENTO UF 2007.xls"

# Carregando dataframe sem os cabeçalhos mesclados da planilha
dados2007 = pd.read_excel(
    caminho,
    header=None,
    skiprows=10,
    na_values=["--"]
)

dados2007

,0,1,2,3,4,5,6,7,8,9,...,49,50,51,52,53,54,55,56,57,58
0,2007,Norte,RO,Rural,Estadual,72.3,57.8,68.5,76.5,80.9,...,10.3,7.6,11.4,9.3,19.9,15.4,12.9,NaN,NaN,17.0
1,2007,Norte,RO,Rural,Federal,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,0.0
2,2007,Norte,RO,Rural,Municipal,95.5,69.7,81.9,86.9,88.7,...,7.1,4.1,8.4,5.9,19.0,11.8,8.3,NaN,11.7,13.8
3,2007,Norte,RO,Rural,Particular,100.0,90.0,92.2,89.4,95.7,...,0.0,0.0,0.3,0.2,4.0,2.9,0.0,NaN,NaN,2.8
4,2007,Norte,RO,Rural,Publico,94.3,68.4,80.7,86.0,88.1,...,7.5,4.4,8.8,6.2,16.9,12.1,9.4,NaN,11.7,13.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
468,2007,Centro_Oeste,DF,Urbana,Total,96.4,89.1,86.0,89.2,92.1,...,3.4,0.8,3.0,1.8,9.6,7.4,6.0,0.0,0.0,8.0
469,Fonte: Mec/Inep/Deed.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
470,"Notas: 1) Inclui as taxas de aprovação, r...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
471,2) Inclui as taxas do Ensino F...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
colunas = list(dados2007.columns)

# Nomeando as 5 primeiras colunas
colunas[:5] = [
    "ANO",
    "REGIAO",
    "UF",
    "LOCALIZACAO",
    "REDE"
]

dados2007.columns = colunas

dados2007

,ANO,REGIAO,UF,LOCALIZACAO,REDE,5,6,7,8,9,...,49,50,51,52,53,54,55,56,57,58
0,2007,Norte,RO,Rural,Estadual,72.3,57.8,68.5,76.5,80.9,...,10.3,7.6,11.4,9.3,19.9,15.4,12.9,NaN,NaN,17.0
1,2007,Norte,RO,Rural,Federal,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,0.0
2,2007,Norte,RO,Rural,Municipal,95.5,69.7,81.9,86.9,88.7,...,7.1,4.1,8.4,5.9,19.0,11.8,8.3,NaN,11.7,13.8
3,2007,Norte,RO,Rural,Particular,100.0,90.0,92.2,89.4,95.7,...,0.0,0.0,0.3,0.2,4.0,2.9,0.0,NaN,NaN,2.8
4,2007,Norte,RO,Rural,Publico,94.3,68.4,80.7,86.0,88.1,...,7.5,4.4,8.8,6.2,16.9,12.1,9.4,NaN,11.7,13.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
468,2007,Centro_Oeste,DF,Urbana,Total,96.4,89.1,86.0,89.2,92.1,...,3.4,0.8,3.0,1.8,9.6,7.4,6.0,0.0,0.0,8.0
469,Fonte: Mec/Inep/Deed.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
470,"Notas: 1) Inclui as taxas de aprovação, r...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
471,2) Inclui as taxas do Ensino F...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Ainda há um rodapé de informações que podem ser descartadas.

In [7]:
# Convertendo ano para numerico.
dados2007["ANO"] = pd.to_numeric(
    dados2007["ANO"],
    errors="coerce"
)

# Filtr os anos que são iguais a 2007, removendo as linhas de rodapé que tem valor diferente a 2007.
dados2007 = dados2007[
    dados2007["ANO"] == 2007
].copy()

dados2007

,ANO,REGIAO,UF,LOCALIZACAO,REDE,5,6,7,8,9,...,49,50,51,52,53,54,55,56,57,58
0,2007.0,Norte,RO,Rural,Estadual,72.3,57.8,68.5,76.5,80.9,...,10.3,7.6,11.4,9.3,19.9,15.4,12.9,NaN,NaN,17.0
1,2007.0,Norte,RO,Rural,Federal,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,0.0
2,2007.0,Norte,RO,Rural,Municipal,95.5,69.7,81.9,86.9,88.7,...,7.1,4.1,8.4,5.9,19.0,11.8,8.3,NaN,11.7,13.8
3,2007.0,Norte,RO,Rural,Particular,100.0,90.0,92.2,89.4,95.7,...,0.0,0.0,0.3,0.2,4.0,2.9,0.0,NaN,NaN,2.8
4,2007.0,Norte,RO,Rural,Publico,94.3,68.4,80.7,86.0,88.1,...,7.5,4.4,8.8,6.2,16.9,12.1,9.4,NaN,11.7,13.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
464,2007.0,Centro_Oeste,DF,Urbana,Estadual,96.1,87.1,83.3,87.1,90.6,...,4.3,0.9,3.7,2.2,12.1,9.9,8.2,NaN,0.0,10.4
465,2007.0,Centro_Oeste,DF,Urbana,Federal,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0
466,2007.0,Centro_Oeste,DF,Urbana,Particular,96.6,98.3,98.9,98.8,98.6,...,0.1,0.2,0.1,0.1,0.2,0.1,0.1,0.0,NaN,0.1
467,2007.0,Centro_Oeste,DF,Urbana,Publico,96.1,87.1,83.3,87.1,90.6,...,4.2,0.9,3.6,2.2,11.9,9.6,8.1,NaN,0.0,10.2


In [8]:
# Localizando posição das colunas de interesse
# Estamos buscando por ANO, UF, REDE, TAXA_APROVACAO, TAXA_REPROVACAO, TAXA_ABANDONO
# Todas as taxas do buscadas são as de ensino médio
dados_brutos = pd.read_excel(caminho, header=None)

for i in range(dados_brutos.shape[1]):
    print(
        f"Coluna {i}:",
        dados_brutos.iloc[6, i],
        "|",
        dados_brutos.iloc[7, i]
    )

Coluna 0: Ano | nan
Coluna 1: Região | nan
Coluna 2: UF | nan
Coluna 3: Localização | nan
Coluna 4: Rede | nan
Coluna 5: Aprovação no 1º Ano do Ensino Fundamental | nan
Coluna 6: Taxa de Aprovação - Ensino Fundamental de 8 e 9 anos | Aprovação na 1ª série/2º Ano
Coluna 7: nan | Aprovação na 2ª série/3º Ano
Coluna 8: nan | Aprovação na 3ª série/4º Ano
Coluna 9: nan | Aprovaçãona 4ª série/5º Ano
Coluna 10: nan | Aprovação na 5ª série/6º Ano
Coluna 11: nan | Aprovação na 6ª série/7º Ano
Coluna 12: nan | Aprovação na 7ª série/8º Ano
Coluna 13: nan | Aprovação na 8ª série/9º Ano
Coluna 14: nan | Aprovação 1ª a 4ª  Série/1º a 5º Ano
Coluna 15: nan | Aprovação 5ª a 8ª Série/ 6º ao 9º Ano
Coluna 16: nan | Total Aprovação Fundamental
Coluna 17: Taxa de Aprovação - Ensino Médio | Aprovação na 1ª série - Médio
Coluna 18: nan | Aprovação na 2ª série - Médio
Coluna 19: nan | Aprovação na 3ª série - Médio
Coluna 20: nan | Aprovação na 4ª série - Médio
Coluna 21: nan | Total  Aprovação Médio Não Seri

Colunas que nos interessam:

Coluna 0: Ano | nan

Coluna 2: UF | nan

Coluna 22: nan | Total  Aprovação Médio

Coluna 40: nan | Total  Reprovação Médio

Coluna 58: nan | Total Abandono - Médio

#Processamento

In [48]:
import pandas as pd

def carregar_rendimento(ano):

    caminho = f"/content/drive/MyDrive/pe-de-meia/inep/TAXAS RENDIMENTO UF {ano}.xls"

    df = pd.read_excel(
        caminho,
        header=None,
        na_values=["--"]
    )

    # Selecionando as colunas relevantes
    colunas = [0, 2, 3, 4, 22, 40, 58]

    # Pegando somente as linhas de dados, sem cabeçalhos de células mescladas
    df = df.iloc[10:, colunas].copy()

    # colocando cabeçalho nomeando as colunas
    df.columns = [
      "ANO",
      "UF",
      "LOCALIZACAO",
      "REDE",
      "TAXA_APROVACAO",
      "TAXA_REPROVACAO",
      "TAXA_ABANDONO"
    ]

    # Convertendo coluna "ANO" para numérico
    df["ANO"] = pd.to_numeric(df["ANO"], errors="coerce")

    df = df[df["ANO"] == ano].copy()

    # Selecionar apenas o total das redes
    df = df[
      (df["REDE"].astype(str).str.strip().str.upper() == "TOTAL") &
      (df["LOCALIZACAO"].astype(str).str.strip().str.upper() == "TOTAL")
    ].copy()

    # Dropar coluna "REDE" e "LOCALIZACAO" já que todas as linhas dela são referentes às taxas totais
    df = df.drop(columns=["REDE", "LOCALIZACAO"])

    taxas = [
        "TAXA_APROVACAO",
        "TAXA_REPROVACAO",
        "TAXA_ABANDONO"
    ]

    for coluna in taxas:
        df[coluna] = pd.to_numeric(df[coluna], errors="coerce")

    df = df.sort_values("UF").reset_index()
    df = df.drop(columns=["index"])

    df.to_csv(
            f"/content/drive/MyDrive/pe-de-meia/inep/inep-data/taxa_rendimento{ano}_uf.csv",
            index=False
        )

    return df

In [49]:
dados2007 = carregar_rendimento(2007)
dados2007

,ANO,UF,TAXA_APROVACAO,TAXA_REPROVACAO,TAXA_ABANDONO
0,2007.0,AC,75.9,9.2,14.9
1,2007.0,AL,71.7,8.4,19.9
2,2007.0,AM,77.9,10.9,11.2
3,2007.0,AP,70.0,10.4,19.6
4,2007.0,BA,68.7,11.4,19.9
5,2007.0,CE,76.7,8.6,14.7
6,2007.0,DF,71.1,20.9,8.0
7,2007.0,ES,77.2,10.3,12.5
8,2007.0,GO,73.6,10.5,15.9
9,2007.0,MA,75.0,7.5,17.5
